### TOTAL ORDER PLANNER



In mathematical terminology, **total order**, **linear order** or **simple order** refers to a set *X* which is said to be totally ordered under &le; if the following statements hold for all *a*, *b* and *c* in *X*:
<br>
If *a* &le; *b* and *b* &le; *a*, then *a* = *b* (antisymmetry).
<br>
If *a* &le; *b* and *b* &le; *c*, then *a* &le; *c* (transitivity).
<br>
*a* &le; *b* or *b* &le; *a* (connex relation).
<br><br>
In simpler terms, a total order plan is a linear ordering of actions to be taken to reach the goal state.
There may be several different total-order plans for a particular goal depending on the problem.
<br>
In the module, the `Linearize` class solves problems using this paradigm.
At its core, the `Linearize` uses a solved planning graph from `GraphPlan` and finds a valid total-order solution for it.


**TOTAL-ORDER PLANNER — Linearize** (architecture & flow)
```text
+---------------------------------------------------------------+
| Input: planning_problem (domain + initial state + goal)       |
+---------------------------------------------------------------+
                |
                v
+---------------------------------------------------------------+
| GraphPlan (builds planning graph & finds level-wise solution) |
+---------------------------------------------------------------+
                |
                v
  (graphPlan_solution: list of action-sets per level)
                |
                v
+---------------------------------------------------------------+
| Linearize.filter(solution)                                    |
| - removes persistence actions (P...)                          |
| - returns filtered list of levels (actions only)              |
+---------------------------------------------------------------+
                |
                v
  For each level in filtered_solution:
    (a set of candidate actions to linearize)
                |
                v
+---------------------------------------------------------------+
| Linearize.orderlevel(level, planning_problem)                 |
| - tries permutations of 'level'                               |
| - for each permutation:                                       |
|     * copy planning_problem -> temp                           |
|     * sequentially temp.act(action)                           |
|     * if any act fails, discard permutation and reset temp    |
| - on success return (valid_permutation, resulting_state_temp) |
+---------------------------------------------------------------+
                |
                v
  (append valid_permutation actions to ordered_solution)
                |
                v
+---------------------------------------------------------------+
| Linearize.execute()                                           |
| - calls GraphPlan.execute()                                   |
| - filters solution                                            | 
| - sequentially orderlevels and accumulates ordered_solution   |
| - returns ordered_solution (total-order plan)                 |
+---------------------------------------------------------------+

Notes interleaved:
- filter: removes persistence actions to focus on real ops.
- orderlevel: brute-force search per level using action execution on temp state.
- execute: composes level-orderings into full linear plan.
- planning_problem is progressively updated via returned temp in orderlevel.
- Complexity: exponential per level due to permutations; relies on small level-size.

In [2]:
from planning import *
import inspect

In [3]:
print(inspect.getsource(Linearize))

class Linearize:

    def __init__(self, planning_problem):
        self.planning_problem = planning_problem

    def filter(self, solution):
        """Filter out persistence actions from a solution"""

        new_solution = []
        for section in solution[0]:
            new_section = []
            for operation in section:
                if not (operation.op[0] == 'P' and operation.op[1].isupper()):
                    new_section.append(operation)
            new_solution.append(new_section)
        return new_solution

    def orderlevel(self, level, planning_problem):
        """Return valid linear order of actions for a given level"""

        for permutation in itertools.permutations(level):
            temp = copy.deepcopy(planning_problem)
            count = 0
            for action in permutation:
                try:
                    temp.act(action)
                    count += 1
                except:
                    count = 0
                    temp = cop

The `filter` method removes the persistence actions (if any) from the planning graph representation.
<br>
The `orderlevel` method finds a valid total-ordering of a specified level of the planning-graph, given the state of the graph after the previous level.
<br>
The `execute` method sequentially calls `orderlevel` for all the levels in the planning-graph and returns the final total-order solution.
<br>
<br>
Let's look at some examples.

In [4]:
# total-order solution for air_cargo problem
Linearize(air_cargo()).execute()

[Load(C1, P1, SFO),
 Load(C2, P2, JFK),
 Fly(P1, SFO, JFK),
 Fly(P2, JFK, SFO),
 Unload(C2, P2, SFO),
 Unload(C1, P1, JFK)]

In [5]:
# total-order solution for spare_tire problem
Linearize(spare_tire()).execute()

[Remove(Spare, Trunk), Remove(Flat, Axle), PutOn(Spare, Axle)]